<a href="https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily': f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

Paste your Hugging Face READ token (hf_...): ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item on one report date (content_id × report_date), drawn from fact_content_daily_performance in the warehouse. I scope to a single mid-panel calendar month, month=2026-03, rather than the final month (June 2026), which is the sealed test window. This differs from the teaching CSV's 'one page, one 90-day snapshot' grain — here the grain is daily, so a page contributes up to ~31 rows within the window, one per report_date

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
q = f"""
SELECT COUNT(*) AS n_rows, MIN(report_date) AS d_min, MAX(report_date) AS d_max
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
"""
con.sql(q).df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,d_min,d_max
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Five features (each "knowable at the decision moment because…"):

1. content_age_days (or bucketed as age_tier) — knowable because it's computed purely from the content's creation date; it advances with the calendar and carries no information about future clicks or impressions.
2. word_count_tier — knowable because article length is fixed once the content is published; it doesn't change based on how the page performs afterward.
3. main_intent — knowable because search intent (informational/transactional/commercial/navigational) is assigned from the target keyword's classification at content-creation time, not from observed outcomes.
4. freshness_tier (from days_since_last_update) — knowable because it only reflects the last edit timestamp, which by construction always precedes the report_date you're scoring against.
5. content_type — knowable because it's set at creation (keyword article / feedly article / comparison article) and never retroactively changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# (1) grain check — should return 0 rows
grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) c
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print('grain violations found:', len(grain_check))

# (2) row count + date span (repeated here for the record)
counts = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS d_min, MAX(report_date) AS d_max
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(counts)

# (3) availability — how many rows survive the GA4 flag filter
avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print(avail)

# Five-feature frame (static content metadata, all knowable before the decision moment)
feat_df = con.sql(f"""
    SELECT
        d.content_hash_id,
        d.content_age_days,
        d.word_count,
        d.main_intent,
        d.days_since_last_update,
        d.content_type,
        f.report_date,
        f.trend_direction  -- kept temporarily for the leakage demo below, drop after
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d USING (content_hash_id)
    WHERE f.month = '2026-03'
""").df()
feat_df['is_declining_label'] = (feat_df['trend_direction'] == 'down').astype(int)
feat_df.head()

# --- Leakage trap: train once WITH the label-derived column, then without ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score
import pandas as pd

work = pd.get_dummies(feat_df.drop(columns=['content_hash_id', 'report_date']), dummy_na=True)

# WITH the leak (trend_direction one-hot columns are derived straight from the label)
X_leaky = work.drop(columns=['is_declining_label'])
y = work['is_declining_label']
Xtr, Xte, ytr, yte = train_test_split(X_leaky, y, test_size=0.25, random_state=42, stratify=y)
m_leaky = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
print('WITH leak (dishonest):', precision_score(yte, m_leaky.predict(Xte)))

# WITHOUT the leak — drop every trend_direction-derived column
safe_cols = [c for c in work.columns if not c.startswith('trend_direction')]
X_safe = work[safe_cols].drop(columns=['is_declining_label'])
Xtr, Xte, ytr, yte = train_test_split(X_safe, y, test_size=0.25, random_state=42, stratify=y)
m_safe = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xtr, ytr)
print('WITHOUT leak (honest):', precision_score(yte, m_safe.predict(Xte)))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain violations found: 0
    n_rows      d_min      d_max
0  9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0


BinderException: Binder Error: Table "d" does not have a column named "content_age_days"

Candidate bindings: : "content_hash_id"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell us intent behind a decline — is_declining_label is a traffic-based proxy, not a true editorial judgment (a page can decline for reasons a refresh won't fix, like seasonality or a SERP feature change). It also can't be read as a single global timeline: per-client history depth varies wildly (dim_clients.gsc_data_start differs per client), so a global calendar window mixes clients with very different maturity. Finally, early rows for any client are GSC-only — GA4 columns are zero-filled before ga4_data_available is TRUE, and that flag can also be NULL rather than FALSE for some clients, so NULL-handling needs explicit filtering, not a blanket zero-fill.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.